# 1. Project Context

This notebook implements a **deep learning benchmark** for cross-subject motor imagery classification on the same task used in the classical baseline notebook.

**Model:** EEGNet (Braindecode)  
**Evaluation:** Subject-wise cross-subject (leave-one-subject-out)  
**Dataset:** BNCI2014_001, left vs right motor imagery only  

The main objective is to quantify overall performance and identify the weakest subjects for targeted improvement in later experiments.

# 2. Environment and Imports

In [1]:
import os

import json

import time

import random

import warnings

import builtins

import inspect

from datetime import datetime

from pathlib import Path



import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns



import torch

import mne



from braindecode.datasets import MOABBDataset

from braindecode.datasets.base import BaseConcatDataset

from braindecode.preprocessing import Preprocessor, preprocess, create_windows_from_events

from braindecode import EEGClassifier



try:

    from braindecode.models import EEGNet

    EEGNET_MODEL_NAME = 'EEGNet'

except Exception:

    from braindecode.models import EEGNetv4 as EEGNet

    EEGNET_MODEL_NAME = 'EEGNetv4'



warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

pd.set_option('display.width', 140)

sns.set_style('whitegrid')



print(f'Using model class: {EEGNET_MODEL_NAME}')

/Users/vadim/Documents/School/Spring 2026/CSCE A662 Advanced Data Mining/Assignments/Assignment 2/bci-inefficiency-analysis/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using model class: EEGNet


In [2]:
import braindecode

print('=' * 60)
print('PACKAGE VERSIONS')
print('=' * 60)
print(f'Python: {os.sys.version.split()[0]}')
print(f'torch: {torch.__version__}')
print(f'mne: {mne.__version__}')
print(f'braindecode: {braindecode.__version__}')
try:
    import moabb
    print(f'moabb: {moabb.__version__}')
except Exception:
    print('moabb: unavailable')

PACKAGE VERSIONS
Python: 3.11.15
torch: 2.10.0
mne: 1.11.0
braindecode: 1.3.2
moabb: 1.4.3


# 3. Experiment Configuration

In [3]:
DATASET_NAME = 'BNCI2014_001'
SUBJECT_IDS = list(range(1, 10))
EVENTS = ['left_hand', 'right_hand']
SFREQ = 250
LOW_CUT_HZ = 8
HIGH_CUT_HZ = 32
TRIAL_START_OFFSET_SEC = 0.0
TRIAL_STOP_OFFSET_SEC = 0.0

BATCH_SIZE = 64
N_EPOCHS = 60
LR = 0.001
WEIGHT_DECAY = 0.0
SEED = 42
DEBUG_SUBJECT_LIMIT = None

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'src' else Path.cwd()
ARTIFACT_DIR = BASE_DIR / 'artifacts' / '03_deep_eegnet'

print('=' * 60)
print('CONFIGURATION')
print('=' * 60)
print(f'Dataset: {DATASET_NAME}')
print(f'Subjects: {SUBJECT_IDS}')
print(f'Events: {EVENTS}')
print(f'Filter band: {LOW_CUT_HZ}-{HIGH_CUT_HZ} Hz')
print(f'Epochs: {N_EPOCHS}, Batch size: {BATCH_SIZE}')
print(f'Learning rate: {LR}, Weight decay: {WEIGHT_DECAY}')
print(f'Device: {DEVICE}')

CONFIGURATION
Dataset: BNCI2014_001
Subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9]
Events: ['left_hand', 'right_hand']
Filter band: 8-32 Hz
Epochs: 60, Batch size: 64
Learning rate: 0.001, Weight decay: 0.0
Device: mps


# 4. Logging and Artifact Setup

In [4]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = ARTIFACT_DIR / 'run.log'

if '_LOG_FILE_HANDLE' in globals() and _LOG_FILE_HANDLE and not _LOG_FILE_HANDLE.closed:
    _LOG_FILE_HANDLE.close()
_LOG_FILE_HANDLE = open(LOG_PATH, 'w', buffering=1, encoding='utf-8')

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop('sep', ' ')
    end = kwargs.pop('end', '\n')
    flush = kwargs.pop('flush', False)
    file = kwargs.pop('file', None)

    message = sep.join(str(arg) for arg in args)
    leading_newlines = len(message) - len(message.lstrip('\n'))
    message_body = message[leading_newlines:]

    def _write_target(text):
        if file is None:
            os.sys.__stdout__.write(text)
            if flush:
                os.sys.__stdout__.flush()
        else:
            file.write(text)
            if flush and hasattr(file, 'flush'):
                file.flush()

    if leading_newlines > 0:
        blanks = '\n' * leading_newlines
        _write_target(blanks)
        _LOG_FILE_HANDLE.write(blanks)

    if message_body:
        ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        stamped = f'[{ts}] {message_body}'
        _write_target(stamped + end)
        _LOG_FILE_HANDLE.write(stamped + end)
    else:
        _write_target(end)
        _LOG_FILE_HANDLE.write(end)

    if flush:
        _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print

print(f'Artifact directory: {ARTIFACT_DIR}')
print(f'Logging to: {LOG_PATH}')

[2026-04-13 18:52:17] Artifact directory: /Users/vadim/Documents/School/Spring 2026/CSCE A662 Advanced Data Mining/Assignments/Assignment 2/bci-inefficiency-analysis/artifacts/03_deep_eegnet
[2026-04-13 18:52:17] Logging to: /Users/vadim/Documents/School/Spring 2026/CSCE A662 Advanced Data Mining/Assignments/Assignment 2/bci-inefficiency-analysis/artifacts/03_deep_eegnet/run.log


In [5]:
def save_plot(filename):
    path = ARTIFACT_DIR / filename
    plt.savefig(path, dpi=300, bbox_inches='tight')
    print(f'Saved plot: {path}')


def save_table(df, filename):
    path = ARTIFACT_DIR / filename
    df.to_csv(path, index=False, encoding='utf-8')
    print(f'Saved table: {path}')


def save_json(payload, filename):
    path = ARTIFACT_DIR / filename
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2)
    print(f'Saved JSON: {path}')

# 5. Dataset Loading and Left/Right Filtering

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

subject_ids = SUBJECT_IDS[:DEBUG_SUBJECT_LIMIT] if DEBUG_SUBJECT_LIMIT else SUBJECT_IDS
dataset = MOABBDataset(dataset_name=DATASET_NAME, subject_ids=subject_ids)

print('=' * 60)
print('DATASET INFORMATION')
print('=' * 60)
print(f'Dataset name: {DATASET_NAME}')
print(f'Number of requested subjects: {len(subject_ids)}')
print(f'Subject IDs: {subject_ids}')

if len(dataset.datasets) == 0:
    raise RuntimeError('Loaded dataset is empty. Check subject IDs and dataset availability.')

example_description = dataset.datasets[0].description
print(f'Example recording description: {dict(example_description)}')

[2026-04-13 18:52:24] ============================================================
[2026-04-13 18:52:24] DATASET INFORMATION
[2026-04-13 18:52:24] ============================================================
[2026-04-13 18:52:24] Dataset name: BNCI2014_001
[2026-04-13 18:52:24] Number of requested subjects: 9
[2026-04-13 18:52:24] Subject IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9]
[2026-04-13 18:52:24] Example recording description: {'subject': 1, 'session': '0train', 'run': '0'}


# 6. Preprocessing and Window Construction

In [7]:
preprocessors = [
    Preprocessor('pick_types', eeg=True, meg=False, stim=False),
    Preprocessor('filter', l_freq=LOW_CUT_HZ, h_freq=HIGH_CUT_HZ),
    Preprocessor(lambda data: data * 1e6, apply_on_array=True),
]

print('Applying preprocessing...')
preprocess(dataset, preprocessors, n_jobs=1)

trial_start_offset_samples = int(TRIAL_START_OFFSET_SEC * SFREQ)
trial_stop_offset_samples = int(TRIAL_STOP_OFFSET_SEC * SFREQ)

print('Creating windows (left/right only)...')
windows_dataset = create_windows_from_events(
    dataset,
    trial_start_offset_samples=trial_start_offset_samples,
    trial_stop_offset_samples=trial_stop_offset_samples,
    window_size_samples=None,
    window_stride_samples=None,
    drop_last_window=False,
    mapping={
        'left_hand': 0,
        'right_hand': 1,
    },
    preload=True,
)

print(f'Number of window datasets: {len(windows_dataset.datasets)}')
print(f'Total number of windows: {len(windows_dataset)}')

[2026-04-13 18:52:24] Applying preprocessing...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 32 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 32.00 Hz
- Upper transition bandwidth: 8.00 Hz (-6 dB cutoff frequency: 36.00 Hz)
- Filter length: 413 samples (1.652 s)

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 32 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method

In [8]:
def _subject_from_description(description):
    if 'subject' in description:
        return int(description['subject'])
    if 'subject_id' in description:
        return int(description['subject_id'])
    raise KeyError('Could not find subject field in dataset description.')


def _extract_targets(concat_ds):
    return np.array([int(concat_ds[i][1]) for i in range(len(concat_ds))], dtype=np.int64)


def _build_eegnet(n_chans, n_times, n_classes):
    sig = inspect.signature(EEGNet)
    params = sig.parameters
    kwargs = {}

    if 'n_chans' in params:
        kwargs['n_chans'] = n_chans
    elif 'in_chans' in params:
        kwargs['in_chans'] = n_chans

    if 'n_outputs' in params:
        kwargs['n_outputs'] = n_classes
    elif 'n_classes' in params:
        kwargs['n_classes'] = n_classes

    if 'n_times' in params:
        kwargs['n_times'] = n_times
    elif 'input_window_samples' in params:
        kwargs['input_window_samples'] = n_times

    if 'final_conv_length' in params:
        kwargs['final_conv_length'] = 'auto'

    if 'drop_prob' in params:
        kwargs['drop_prob'] = 0.25

    if 'sfreq' in params:
        kwargs['sfreq'] = SFREQ

    return EEGNet(**kwargs)

# 7. Cross-Subject EEGNet Evaluation Setup

In [9]:
subject_to_datasets = {}
for ds in windows_dataset.datasets:
    subj = _subject_from_description(ds.description)
    subject_to_datasets.setdefault(subj, []).append(ds)

subject_keys = sorted(subject_to_datasets.keys())
print('=' * 60)
print('CROSS-SUBJECT SETUP')
print('=' * 60)
print(f'Available subjects in windows: {subject_keys}')
for subj in subject_keys:
    fold_windows = sum(len(ds) for ds in subject_to_datasets[subj])
    print(f'Subject {subj}: {fold_windows} windows')

[2026-04-13 18:52:30] ============================================================
[2026-04-13 18:52:30] CROSS-SUBJECT SETUP
[2026-04-13 18:52:30] ============================================================
[2026-04-13 18:52:30] Available subjects in windows: [1, 2, 3, 4, 5, 6, 7, 8, 9]
[2026-04-13 18:52:30] Subject 1: 288 windows
[2026-04-13 18:52:30] Subject 2: 288 windows
[2026-04-13 18:52:30] Subject 3: 288 windows
[2026-04-13 18:52:30] Subject 4: 288 windows
[2026-04-13 18:52:30] Subject 5: 288 windows
[2026-04-13 18:52:30] Subject 6: 288 windows
[2026-04-13 18:52:30] Subject 7: 288 windows
[2026-04-13 18:52:30] Subject 8: 288 windows
[2026-04-13 18:52:30] Subject 9: 288 windows


# 8. Run Subject-Wise Deep Learning Benchmark

In [ ]:
results_rows = []



for held_out_subject in subject_keys:

    print('\n' + '=' * 60)

    print(f'HELD-OUT SUBJECT: {held_out_subject}')

    print('=' * 60)



    train_datasets = []

    test_datasets = []



    for subj, ds_list in subject_to_datasets.items():

        if subj == held_out_subject:

            test_datasets.extend(ds_list)

        else:

            train_datasets.extend(ds_list)



    if len(train_datasets) == 0 or len(test_datasets) == 0:

        print('Skipping fold due to empty train or test split.')

        continue



    train_full = BaseConcatDataset(train_datasets)

    test_set = BaseConcatDataset(test_datasets)

    y_test = _extract_targets(test_set)



    n_chans, n_times = train_full[0][0].shape[0], train_full[0][0].shape[-1]

    model = _build_eegnet(n_chans=n_chans, n_times=n_times, n_classes=2)



    clf = EEGClassifier(

        model,

        criterion=torch.nn.CrossEntropyLoss,

        optimizer=torch.optim.Adam,

        optimizer__lr=LR,

        optimizer__weight_decay=WEIGHT_DECAY,

        batch_size=BATCH_SIZE,

        max_epochs=N_EPOCHS,

        train_split=None,

        device=DEVICE,

        iterator_train__shuffle=True,

        verbose=0,

    )



    start_t = time.time()

    clf.fit(train_full, y=None)

    fit_time = time.time() - start_t



    y_pred = clf.predict(test_set)

    score = float((y_pred == y_test).mean())



    print(f'Train samples: {len(train_full)}')

    print(f'Test samples: {len(test_set)}')

    print(f'Fit time (sec): {fit_time:.2f}')

    print(f'Held-out accuracy: {score:.4f}')



    results_rows.append({

        'subject': held_out_subject,

        'score': score,

        'train_samples': len(train_full),

        'test_samples': len(test_set),

        'fit_time_sec': fit_time,

        'pipeline': 'EEGNet',

        'dataset': DATASET_NAME,

    })



results = pd.DataFrame(results_rows)

if results.empty:

    raise RuntimeError('No fold results were produced. Please inspect dataset splits and training logs.')



print('\nBenchmark run complete.')

print(f'Results shape: {results.shape}')


[2026-04-13 18:52:30] ============================================================
[2026-04-13 18:52:30] HELD-OUT SUBJECT: 1
[2026-04-13 18:52:30] ============================================================
  epoch    train_loss     dur
-------  ------------  ------
      1        0.6958  7.0000
      2        0.6642  2.6701
      3        0.5851  2.4091
      4        0.5432  2.3667
      5        0.5141  2.3894
      6        0.4943  2.3729
      7        0.4973  2.4514
      8        0.4923  2.4544
      9        0.4860  2.4413
     10        0.4788  2.3750
     11        0.4732  2.4182
     12        0.4803  2.4336
     13        0.4683  2.4075
     14        0.4725  2.3853
     15        0.4592  2.3538
     16        0.4615  2.4263
     17        0.4489  2.4206
     18        0.4597  2.4102
     19        0.4484  2.3748
     20        0.4559  2.4247
     21        0.4566  2.4460
     22        0.4349  2.3142
     23        0.4333  2.4173
     24        0.4468  2.4513
     25    

# 9. Inspect Results

In [ ]:
print('=' * 60)
print('RESULTS HEAD')
print('=' * 60)
print(results.head())

summary_metrics = pd.DataFrame([
    {
        'pipeline': 'EEGNet',
        'mean_score': float(results['score'].mean()),
        'std_score': float(results['score'].std(ddof=1)),
        'min_score': float(results['score'].min()),
        'max_score': float(results['score'].max()),
    }
])

print('\n' + '=' * 60)
print('RESULTS SUMMARY')
print('=' * 60)
print(f"Mean score: {summary_metrics.loc[0, 'mean_score']:.4f}")
print(f"Std score: {summary_metrics.loc[0, 'std_score']:.4f}")
print(f"Min score: {summary_metrics.loc[0, 'min_score']:.4f}")
print(f"Max score: {summary_metrics.loc[0, 'max_score']:.4f}")

# 10. Per-Subject Analysis

In [ ]:
per_subject_scores = results.groupby('subject', as_index=False)['score'].mean().sort_values('score')

print('=' * 60)
print('PER-SUBJECT PERFORMANCE (sorted from worst to best)')
print('=' * 60)
print(per_subject_scores.to_string(index=False))

best_row = per_subject_scores.iloc[-1]
worst_row = per_subject_scores.iloc[0]

print(f"\nBest performing subject: {int(best_row['subject'])} ({best_row['score']:.4f})")
print(f"Worst performing subject: {int(worst_row['subject'])} ({worst_row['score']:.4f})")

bottom_subjects = per_subject_scores.head(3)
median_score = float(per_subject_scores['score'].median())
below_median = int((per_subject_scores['score'] < median_score).sum())

print('\nBottom 3 subjects:')
for _, row in bottom_subjects.iterrows():
    print(f"  Subject {int(row['subject'])}: {row['score']:.4f}")

print(f'Median score: {median_score:.4f}')
print(f'Subjects below median: {below_median}/{len(per_subject_scores)}')

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(
    per_subject_scores['subject'].astype(str),
    per_subject_scores['score'],
    color='steelblue',
    edgecolor='black'
)
plt.axhline(y=0.5, color='red', linestyle='--', linewidth=1, label='Chance level')
plt.xlabel('Subject ID', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Cross-Subject Deep Performance (EEGNet)', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
save_plot('per_subject_accuracy_bar.png')
plt.show()

plt.figure(figsize=(8, 5))
plt.boxplot([per_subject_scores['score'].values], labels=['EEGNet'])
plt.axhline(y=0.5, color='red', linestyle='--', linewidth=1, label='Chance level')
plt.ylabel('Accuracy', fontsize=12)
plt.title('Deep Model Performance Distribution', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
save_plot('performance_distribution_boxplot.png')
plt.show()

# 11. Save Artifacts

In [ ]:
save_table(results, 'results.csv')
save_table(per_subject_scores, 'per_subject_scores.csv')
save_table(summary_metrics, 'summary_metrics.csv')

summary_payload = summary_metrics.iloc[0].to_dict()
summary_payload['bottom_3_subjects'] = [int(s) for s in bottom_subjects['subject'].tolist()]
summary_payload['median_score'] = median_score
summary_payload['subjects_below_median'] = below_median
save_json(summary_payload, 'summary_metrics.json')

print('All required artifacts were saved successfully.')

# 12. Interpretation and Next Step

This notebook established a deep-learning benchmark using EEGNet on the same left-vs-right BNCI2014_001 task used in the classical baseline.

The results show subject-to-subject variability, with several subjects consistently harder to classify in the cross-subject setting. These worst-performing subjects are strong candidates for targeted follow-up improvements.

The next notebook can directly compare EEGNet against CSP+LDA and test subject-focused strategies to improve generalization on the most challenging participants.